# 01 — Reconstruction Demo

Sanity-check that DUSt3R reconstruction works end-to-end on a handful of
images before building anything on top of it. Run `00_setup_colab.ipynb`
first.


In [ ]:
import sys
sys.path.append('.')
from ugap.reconstruction import DUSt3RWrapper
from ugap.utils.visualization import plot_uncertainty_pointcloud
import numpy as np


## Get some test images

Easiest path: generate a synthetic multi-view sequence with PyBullet
(zero download), or upload 3-5 photos of a real object taken from
different angles.


In [ ]:
# Option A: synthetic object via PyBullet (fast, no download)
from ugap.data.sim_generator import generate_dataset
generate_dataset(out_root='data/sim_sequences', n_objects_per_class=1, n_views=6, image_size=256, seed=1)

import glob
image_paths = sorted(glob.glob('data/sim_sequences/obj_0000_opaque/views/view_*.png'))
print(f"{len(image_paths)} images:", image_paths)


In [ ]:
# Option B: upload your own images instead (uncomment to use)
# from google.colab import files
# uploaded = files.upload()
# image_paths = sorted(uploaded.keys())


In [ ]:
wrapper = DUSt3RWrapper(
    checkpoint='checkpoints/dust3r/DUSt3R_ViT-L_512.pth',
    device='cuda',
    image_size=512,
)

recon = wrapper.reconstruct(image_paths)
print("Points:", recon.points.shape)
print("Confidence range:", recon.confidence.min(), recon.confidence.max())
print("Camera poses:", recon.per_image_poses.shape)


In [ ]:
# Visualize the DUSt3R-native confidence (not our evidential uncertainty yet --
# that comes in notebook 02) as a first sanity check.
fig = plot_uncertainty_pointcloud(recon.points, recon.confidence, title="DUSt3R native confidence")
fig


If this looks like a reasonable point cloud with confidence roughly
tracking well-observed vs. sparsely-observed regions, move on to
`02_uncertainty_head_training.ipynb`.
